In [1]:
import sys
sys.path.append("/axovol/l1ad/src")  # project root
### Imports
import json
import numpy as np
from torch.utils import data
import yaml
import h5py
import torch
import trainer
import model


In [2]:
def load_config(config_path, overrides=None):
    """Load YAML config and apply CLI overrides."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    if overrides:
        for key, value in overrides.items():
            keys = key.split(".")
            sub = config
            for k in keys[:-1]:
                sub = sub.setdefault(k, {})
            sub[keys[-1]] = value
    return config

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cpu')

In [5]:
f = h5py.File('../training/v5/conditionsupdate_apr25.h5', 'r')

In [6]:
x_train = f['data']["Background_data"]["Train"]["DATA"][:50000]
x_test = f['data']["Background_data"]["Test"]["DATA"][:50000]
x_sig = f['data']["Signal_data"]["GluGluHToBB_M-125"]["DATA"][:50000]

scale = f['data']["Normalisation"]["norm_scale"][:]
bias = f['data']["Normalisation"]["norm_bias"][:]

In [7]:
x_train = torch.tensor(np.reshape(x_train,(x_train.shape[0],-1))).to(torch.float32).to(device)
x_test = torch.tensor(np.reshape(x_test,(x_test.shape[0],-1))).to(torch.float32).to(device)
x_sig = torch.tensor(np.reshape(x_sig,(x_sig.shape[0],-1))).to(torch.float32).to(device)


In [8]:
batch_size = 2048

train_loader = data.DataLoader(
            dataset=data.TensorDataset(x_train),
            batch_size=batch_size,
        )

val_loader = data.DataLoader(
            dataset=data.TensorDataset(x_test),
            batch_size=batch_size,
        )

val_loader_no_batch = data.DataLoader(
    dataset=data.TensorDataset(x_test),
    batch_size=len(x_train),
)

sig_loader = data.DataLoader(
            dataset=data.TensorDataset(x_sig),
            batch_size=batch_size,
        )

In [9]:
class MyLoader():
    def __init__(self, train_loader, val_loader, val_loader_no_batch, ood_loader) -> None:
        self.training_loader = train_loader
        self.validation_loader = val_loader
        self.validation_loader_no_batch = val_loader_no_batch
        self.ood_loader = ood_loader
        
loaders = MyLoader(train_loader, val_loader, val_loader_no_batch, sig_loader)

In [10]:
config = yaml.safe_load(open("config/config.yaml", "r"))
config['training']["batch_size"] = batch_size
config['training']['n_epochs'] = 5

In [11]:
config

{'data': {'filepath': '../training/v5/conditionsupdate_apr25.h5',
  'output': 'output_full_10_21',
  'standardize': False,
  'min_max': True,
  'n_train_sample': 50000,
  'n_test_sample': 10000},
 'training': {'batch_size': 2048,
  'es_patience': 5000,
  'n_epochs': 5,
  'optimizer': 'AdamW',
  'learning_rate': 0.005,
  'lr_scheduler': 'ReduceLROnPlateau',
  'lr_scheduler_args': {'mode': 'min',
   'factor': 0.8,
   'patience': 20,
   'threshold': 0.001,
   'cooldown': 10,
   'threshold_mode': 'rel',
   'min_lr': 0}},
 'wnae': {'sampling': 'pcd',
  'x_step': 5,
  'x_step_size': 0.05,
  'x_noise_std': 0.22,
  'x_temperature': 0.063,
  'x_bound': [-3, 3],
  'x_clip_grad': None,
  'x_reject_boundary': False,
  'x_mh': False,
  'z_step': 5,
  'z_step_size': 0.05,
  'z_temperature': 0.063,
  'z_noise_std': 1,
  'z_bound': None,
  'z_clip_grad': None,
  'z_reject_boundary': False,
  'z_mh': False,
  'spherical': False,
  'initial_dist': 'gaussian',
  'replay': True,
  'replay_ratio': 0.95,
  

In [ ]:
USE_VICREG = True
VICREG_CHECKPOINT = "/axovol/l1ad/checkpoints/vicreg_fixed/checkpoint_epoch1000.pt"

if USE_VICREG:
    vic_cfg = yaml.safe_load(open("config/vicreg_config.yaml"))
    enc_cfg = vic_cfg["model"]["encoder"]
    embed_dim = enc_cfg["bottleneck_size"]

    vicreg_enc = model.Encoder(
        input_size=57,
        intermediate_architecture=enc_cfg["intermediate_architecture"],
        bottleneck_size=embed_dim,
        drop_out=enc_cfg["drop_out"],
    )
    ckpt = torch.load(VICREG_CHECKPOINT, map_location=device, weights_only=False)
    enc_state = {
        k[len("encoder."):]: v
        for k, v in ckpt["model_state_dict"].items()
        if k.startswith("encoder.")
    }
    vicreg_enc.load_state_dict(enc_state)
    vicreg_enc = vicreg_enc.to(device).eval()
    print(f"Loaded VICReg encoder (epoch {ckpt.get('epoch', '?')})")
    print(f"  57 → {enc_cfg['intermediate_architecture']} → {embed_dim}")

    # precompute embeddings once — VICReg encoder is frozen
    with torch.no_grad():
        e_train = vicreg_enc(x_train)
        e_test  = vicreg_enc(x_test)
        e_sig   = vicreg_enc(x_sig)
    print(f"Embeddings: train={tuple(e_train.shape)}, test={tuple(e_test.shape)}, sig={tuple(e_sig.shape)}")

    # DataLoaders over embeddings
    emb_train_loader = data.DataLoader(data.TensorDataset(e_train), batch_size=batch_size, shuffle=True)
    emb_val_loader   = data.DataLoader(data.TensorDataset(e_test),  batch_size=batch_size)
    emb_val_no_batch = data.DataLoader(data.TensorDataset(e_test),  batch_size=len(e_test))
    emb_sig_loader   = data.DataLoader(data.TensorDataset(e_sig),   batch_size=batch_size)
    emb_loaders = MyLoader(emb_train_loader, emb_val_loader, emb_val_no_batch, emb_sig_loader)


In [ ]:
import trainer as trainer_module

if USE_VICREG:
    # WNAE operates on embed_dim-dimensional VICReg embeddings
    wnae_bottleneck = 4
    wnae_encoder = model.Encoder(
        input_size=embed_dim,
        intermediate_architecture=[8],
        bottleneck_size=wnae_bottleneck,
        drop_out=None,
    ).to(device)
    wnae_decoder = model.Decoder(
        output_size=embed_dim,
        intermediate_architecture=[8],
        bottleneck_size=wnae_bottleneck,
        drop_out=None,
    ).to(device)

    wnae_config = yaml.safe_load(open("config/config.yaml"))
    wnae_config['training']['batch_size'] = batch_size
    wnae_config['training']['n_epochs'] = 5
    wnae_config['model']['encoder']['bottleneck_size'] = wnae_bottleneck
    wnae_config['model']['decoder']['bottleneck_size'] = wnae_bottleneck

    wnae_trainer = trainer_module.TrainerWassersteinNormalizedAutoEncoder(
        config=wnae_config,
        loader=emb_loaders,
        encoder=wnae_encoder,
        decoder=wnae_decoder,
        device=device,
        output_path="output_vicreg_wnae",
        loss_function="wnae",
    )
    wnae_trainer.train()
    wnae_trainer.save_train_plot()


In [15]:
input_size = x_train.shape[-1]
encoder = model.Encoder(input_size=input_size, 
                        intermediate_architecture=config['model']['encoder']['intermediate_architecture'],
                        bottleneck_size=config['model']['encoder']['bottleneck_size'],
                        drop_out=config['model']['encoder']['drop_out']
)   

decoder = model.Decoder(
    output_size=input_size,
    intermediate_architecture=config['model']['decoder']['intermediate_architecture'],
    bottleneck_size=config['model']['decoder']['bottleneck_size'],
    drop_out=config['model']['decoder']['drop_out']
)
encoder = encoder.to(device)
decoder = decoder.to(device)

trainer = trainer.TrainerWassersteinNormalizedAutoEncoder(
    config=config,
    loader=loaders,
    encoder=encoder,
    decoder=decoder,
    device=device,
    output_path=config['data']['output'],
    loss_function="wnae",  # can change to "ae" or "nae"
)

trainer.train()
log.info("Saving...")
trainer.save_train_plot()
log.info("Done.")

[INFO] Starting model fitting
[INFO] 
Epoch 0/33 Training
 12%|█▏        | 3/25 [02:51<20:58]


KeyboardInterrupt: 